In [4]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="phi3:mini",
    base_url="http://localhost:11434",
    temperature=0.1,
    num_gpu=0
)

# Simple direct call — should respond in <30 seconds
response = llm.invoke("Reply with one word: Hello")
print(f"✅ Ollama responded: {response}")

✅ Ollama responded: Hello.


In [5]:
from ragas.llms import LangchainLLMWrapper
from langchain_core.prompt_values import StringPromptValue

wrapped = LangchainLLMWrapper(llm)

# Test if RAGAS wrapper can call the LLM
response = await wrapped.generate(
    prompts=[StringPromptValue(text="Reply with one word: Hello")]
)
print(f"✅ RAGAS wrapper responded: {response}")

TypeError: BaseRagasLLM.generate() got an unexpected keyword argument 'prompts'

In [6]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface import HuggingFaceEmbeddings

# Minimal single metric test
data = Dataset.from_dict({
    "question": ["What is Python?"],
    "answer": ["Python is a programming language."],
    "contexts": [["Python is a high-level programming language used for many purposes."]],
    "ground_truth": ["Python is a programming language."]
})

hf_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

try:
    results = evaluate(
        data,
        metrics=[answer_relevancy],  # just one metric
        llm=LangchainLLMWrapper(llm),
        embeddings=LangchainEmbeddingsWrapper(hf_embeddings),
        raise_exceptions=True       # show actual error instead of nan
    )
    print(f"✅ answer_relevancy: {results['answer_relevancy']}")
except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {e}")

Evaluating: 100%|████████████████████████████████████████████████████████████████████████| 1/1 [00:28<00:00, 28.74s/it]


✅ answer_relevancy: [0.7855399688880498]
